In [1]:
import networkx as nx
import numpy as np
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
import urllib.request as urlrequest
from IPython.display import Image
from collections import Counter
from goose3 import Goose
from goose3.network import NetworkError
from bertopic import BERTopic
from scipy.cluster import hierarchy as sch
from sklearn.feature_extraction.text import CountVectorizer
from urllib.parse import urlparse
from scipy.spatial.distance import pdist, squareform


In [7]:
mbfcs = pd.read_csv('data/mbfc_source_data.csv')
print(mbfcs.head()) # Display the first few rows of the DataFrame

                    name              display_name  \
0    100daysinappalachia    100 Days in Appalachia   
1       100milefreepress       100 Mile Free Press   
2              100%fedup               100% Fed up   
3  koln\kgin-lincolnnews  KOLN\KGIN - Lincoln News   
4   knpl-northplattenews  KNPL - North Platte News   

                                            mbfc_url  \
0  https://mediabiasfactcheck.com/100-days-in-app...   
1  https://mediabiasfactcheck.com/100-mile-free-p...   
2  https://mediabiasfactcheck.com/100-percent-fed...   
3  https://mediabiasfactcheck.com/kolnkgin-lincol...   
4  https://mediabiasfactcheck.com/knpl-north-plat...   

                                website        country  \
0  https://www.100daysinappalachia.com/  United States   
1                  100milefreepress.net         Canada   
2          https://100percentfedup.com/  United States   
3              https://www.1011now.com/  United States   
4                    http://1011np.com/  United 

In [8]:
def clean(df):
    values_to_keep = [193,190,10,12,173,90,112,145,20,40,111,43,42,51,141]
    df = df.rename(columns=map)
    df = df[(df['Actor1Geo_CountryCode'] == 'US') | (df['Actor2Geo_CountryCode'] == 'US') | (df['ActionGeo_CountryCode'] == 'US')]
    df = df[df['EventCode'].isin(values_to_keep)]
    df = df.drop(columns=['MonthYear', 'FractionDate','Actor1Code', 'Actor1Name','Actor1CountryCode', 'Actor1KnownGroupCode','Actor1EthnicCode', 'Actor1Religion1Code','Actor1Religion2Code', 'Actor1Type1Code','Actor1Type2Code','Actor1Type3Code','Actor2Code', 'Actor2Name','Actor2CountryCode', 'Actor2KnownGroupCode','Actor2EthnicCode', 'Actor2Religion1Code','Actor2Religion2Code', 'Actor2Type1Code','Actor2Type2Code','Actor2Type3Code','Actor1Geo_Type', 'Actor1Geo_Fullname','Actor1Geo_ADM1Code', 'Actor1Geo_Lat','Actor1Geo_Long', 'Actor1Geo_FeatureID', 'Actor2Geo_Type', 'Actor2Geo_Fullname', 'Actor2Geo_ADM1Code', 'Actor2Geo_Lat','Actor2Geo_Long','Actor2Geo_FeatureID'], axis=1)
    df = df[(df['AvgTone'] < 1)]
    return df

def simplify(domain):
    parts = domain.split('.')
    # Remove common subdomain prefixes like 'www'
    if parts[0] == 'www':
        parts = parts[1:]
    # Heuristic: use the first part before the main TLD (.com, .org, etc.)
    return parts[-3] if len(parts) > 2 else parts[0]

In [9]:
f_20171031_NY = pd.read_csv('f_20171031_NY.csv')
f_20170812_VA = pd.read_csv('f_20170812_VA.csv')
f_20160612_FL = pd.read_csv('f_20160612_FL.csv')
f_20140524_CA = pd.read_csv('f_20140524_CA.csv')
f_20250602_CO = pd.read_csv('f_20250602_CO.csv')

In [18]:
def replace_nans(text_list, replacement=""):
    return ["" if str(x).strip().lower() == "nan" or x != x else x for x in text_list]

def target(df, mbfcs):
    df['factual_reporting'] = ''  # Initialize column as object type
    df['factual_reporting'] = df['factual_reporting'].astype('object')
    # Normalize mbfcs keys for matching
    mbfcs['normalized_display_name'] = mbfcs['display_name'].str.lower().str.replace(" ", "", regex=False)
    mbfcs['normalized_website'] = mbfcs['website'].apply(lambda x: urlparse(x).hostname if pd.notnull(x) else '').str.lower()

    for i in df.index:
        
        target_clean = str(df.loc[i, 'target']).lower().replace(" ", "")
        domain_clean = str(df.loc[i, 'domain']).lower().replace(" ", "")

        match = mbfcs[mbfcs['normalized_display_name'] == target_clean]
        if not match.empty:
            df.at[i, 'factual_reporting'] = match.iloc[0]['factual_reporting']
            continue

        match = mbfcs[mbfcs['normalized_website'] == domain_clean]
        if not match.empty:
            df.at[i, 'factual_reporting'] = match.iloc[0]['factual_reporting']
        else:
            df.at[i, 'factual_reporting'] = None
    df = df.dropna(axis=0, subset=['text'])
    return df


In [19]:
f_20171031_NY = target(f_20171031_NY, mbfcs)
f_20170812_VA = target(f_20170812_VA, mbfcs)
f_20160612_FL = target(f_20160612_FL, mbfcs)
f_20250602_CO = target(f_20250602_CO, mbfcs)
f_20140524_CA = target(f_20140524_CA, mbfcs)

In [20]:
result_series = set(f_20140524_CA.loc[f_20140524_CA['factual_reporting'].isnull(), 'domain']) | \
                set(f_20171031_NY.loc[f_20171031_NY['factual_reporting'].isnull(), 'domain'])| \
                set(f_20170812_VA.loc[f_20170812_VA['factual_reporting'].isnull(), 'domain'])| \
                set(f_20160612_FL.loc[f_20160612_FL['factual_reporting'].isnull(), 'domain'])| \
                set(f_20250602_CO.loc[f_20250602_CO['factual_reporting'].isnull(), 'domain'])

In [21]:
result_series

{'101touchfm.co.uk',
 '1055online.iheart.com',
 '790talknow.com',
 '937now.iheart.com',
 '95rockfm.com',
 '991thewhale.com',
 '999thepoint.com',
 'adage.com',
 'afr.net',
 'appleinsider.com',
 'archive.thinkprogress.org',
 'archiwum.thenews.pl',
 'article.wn.com',
 'au.news.yahoo.com',
 'augustafreepress.com',
 'axisoflogic.com',
 'bgr.com',
 'biztoc.com',
 'blog.timesunion.com',
 'bloody-disgusting.com',
 'bob949.iheart.com',
 'bonhamjournal.com',
 'brobible.com',
 'brooklyneagle.com',
 'brudirect.com',
 'bwog.com',
 'catholicphilly.com',
 'collider.com',
 'collive.com',
 'colombogazette.com',
 'commercialobserver.com',
 'dailythepatriot.com',
 'dailytimes.com.pk',
 'dailyvoice.com',
 'datechguyblog.com',
 'dnd.com.pk',
 'dominicanrepublicpost.com',
 'dunyanews.tv',
 'en.aswatmasriya.com',
 'en.trend.az',
 'english.news.cn',
 'entertainment.inquirer.net',
 'extratv.com',
 'finance.yahoo.com',
 'financialtribune.com',
 'finanza.repubblica.it',
 'flaglerlive.com',
 'founderscode.com',
 

In [17]:
f_20140524_CA

,GlobalEventID,date,Year,IsRootEvent,EventCode,EventBaseCode,EventRootCode,QuadClass,GoldsteinScale,NumMentions,...,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL,domain,target,title,text,description,factual_reporting
0,298548627,20140517,2014,1,12,12,1,1,-0.4,2,...,-119.746,CA,20140524,http://www.cbsnews.com/news/elliot-rodger-cali...,www.cbsnews.com,cbsnews,Shooting spree suspect son of longtime Hollywo...,"Elliot Rodger, the suspect in Friday's deadly ...","A life of ""sadness, anger and hatred,"" accordi...",High
1,298551064,20140524,2014,1,173,173,17,4,-5.0,3,...,-119.746,CA,20140524,http://news.yahoo.com/farmers-death-still-myst...,news.yahoo.com,news,"Yahoo News: Latest and Breaking News, Headline...",NaN,The latest news and headlines from Yahoo News....,High
2,298551970,20140524,2014,1,193,193,19,4,-10.0,6,...,-119.746,CA,20140524,http://www.thesundaytimes.co.uk/sto/news/irela...,www.thetimes.com,thetimes,Latest news & breaking headlines,NaN,"The latest breaking UK, US, world, business an...",NaN
3,298554304,20140524,2014,1,190,190,19,4,-10.0,46,...,-119.746,CA,20140524,http://www.military.com/daily-news/2014/05/22/...,www.military.com,military,Military Daily News,NaN,Daily updates of everything that you need know...,High
4,298556473,20140524,2014,1,190,190,19,4,-10.0,116,...,-119.746,CA,20140524,http://www.dw.de/california-gunman-was-hollywo...,www.dw.com,dw,US gunman 'was director's son' – DW – 05/25/2014,A Hollywood director says he believes his son ...,A Hollywood director says he believes his son ...,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,298634860,20140524,2014,1,193,193,19,4,-10.0,5,...,-119.746,CA,20140524,http://zeenews.india.com/news/world/seven-dead...,zeenews.india.com,zeenews,"Seven dead, several injured in California coll...",Los Angeles: Seven people were killed and seve...,Seven people were killed and seven others woun...,None
78,298635073,20140524,2014,0,190,190,19,4,-10.0,1,...,-119.746,CA,20140524,http://www.ibtimes.com/uc-santa-barbara-shooti...,www.ibtimes.com,ibtimes,UC Santa Barbara Shooting Update: 3 Stabbing V...,"Update 9:20 p.m. EDT: Elliot Rodger, the alleg...",The gunman allegedly began his murderous spree...,Mixed
79,298635186,20140524,2014,0,10,10,1,1,0.0,20,...,-119.746,CA,20140524,http://www.odt.co.nz/news/world/303498/six-kil...,www.odt.co.nz,odt,"'Madman' killer stabs three, shoots three",A gunman described as severely mentally distur...,A gunman described as severely mentally distur...,None
80,298636264,20140524,2014,1,10,10,1,1,0.0,6,...,-119.746,CA,20140524,http://time.com/112292/states-running-out-of-w...,time.com,time,These 7 States Are Running Out of Water,By Alexander E.M. Hess and Thomas C. Frohlich\...,This post is in partnership with 24/7Wall Stre...,High
